# Model Training — Predicting Smartphone Addiction

Step-by-step walkthrough of the modeling pipeline behind `scripts/train_ensemble.py`,
the approach that produced the results in the README:

| Model | OOF AUC |
|---|---|
| LightGBM | 0.9637 |
| XGBoost | 0.9641 |
| **Stacked ensemble** | **0.9642** |

This notebook runs the *exact same* pipeline — same feature engineering, same
hyperparameters, same 5-fold stratified CV, same random seed — cell by cell, so each stage
(splitting, feature engineering, per-fold training, stacking, evaluation) is visible and
inspectable on its own rather than buried in one script.

See `notebooks/eda.ipynb` / `eda_plotly.ipynb` for the exploratory analysis that motivated
these choices.

**Runtime:** training 2 models × 5 folds on ~690K rows takes roughly **30 minutes** on a
typical laptop CPU — most of the notebook's wall-clock time is the two "Train … — fold"
cells further down.

## 1. Setup

In [1]:
import time
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42
N_SPLITS = 5

DATA_DIR = "../data"
OUT_DIR = "../submissions"

NUM_COLS = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
TARGET = "addicted_label"

print(f"lightgbm {lgb.__version__}, xgboost {xgb.__version__}")

lightgbm 4.6.0, xgboost 3.2.0


## 2. Load the data

`train.csv` is the full labeled training set from Kaggle; `test.csv` has no
`addicted_label` column — that's what we produce predictions for at the end.

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"train shape: {train.shape}")
print(f"test shape:  {test.shape}")
train.head()

train shape: (691369, 14)
test shape:  (296302, 13)


,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [3]:
y = train[TARGET].values
test_ids = test["id"].values

print(f"target balance: {pd.Series(y).value_counts(normalize=True).round(3).to_dict()}")

target balance: {1: 0.709, 0: 0.291}


## 3. Train/validation split

Two different splitting strategies show up in this pipeline, for two different purposes:

1. **A single hold-out split** (`train_test_split`) — the classic way to get one
   validation set quickly. Useful for fast iteration, but a single split means the AUC
   estimate depends on which rows happened to land in the validation fold.
2. **5-fold stratified cross-validation** — what the final model actually uses. Every row
   gets used for validation exactly once (across the 5 folds), which gives a much more
   stable estimate of generalization AUC and, as a side effect, produces **out-of-fold
   (OOF) predictions** for the full training set — needed later to fit the stacking
   meta-model without leakage.

Both are stratified on the target so the ~71%/29% class balance is preserved in every
split.

In [4]:
# 3a. Single hold-out split — illustrates the basic mechanics of train/test splitting.
X_demo_train, X_demo_val, y_demo_train, y_demo_val = train_test_split(
    train.drop(columns=["id", TARGET]), y,
    test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)
print(f"hold-out split: {len(X_demo_train):,} train rows / {len(X_demo_val):,} validation rows")
print(f"train target rate: {y_demo_train.mean():.4f}   val target rate: {y_demo_val.mean():.4f}")
del X_demo_train, X_demo_val, y_demo_train, y_demo_val  # only needed for the illustration above

hold-out split: 553,095 train rows / 138,274 validation rows
train target rate: 0.7094   val target rate: 0.7094


In [5]:
# 3b. 5-fold StratifiedKFold — this is what the final model trains on.
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(train, y))

for i, (tr_idx, va_idx) in enumerate(folds):
    print(f"fold {i+1}: {len(tr_idx):,} train / {len(va_idx):,} val "
          f"(val target rate {y[va_idx].mean():.4f})")

fold 1: 553,095 train / 138,274 val (val target rate 0.7094)
fold 2: 553,095 train / 138,274 val (val target rate 0.7094)
fold 3: 553,095 train / 138,274 val (val target rate 0.7094)
fold 4: 553,095 train / 138,274 val (val target rate 0.7094)
fold 5: 553,096 train / 138,273 val (val target rate 0.7094)


## 4. Feature engineering

Built from the EDA findings: missingness is spread across every column (so missing
indicators help trees route around imputation), and usage-*intensity* and usage-*ratio*
features (notifications, app opens, weekend vs. weekday screen time) carry more signal
than the raw hours features alone. Each block below is applied to train and test
identically.

In [6]:
def add_missing_indicators(df):
    df = df.copy()
    for c in NUM_COLS + CAT_COLS:
        df[f"{c}_missing"] = df[c].isnull().astype(np.int8)
    df["n_missing"] = df[NUM_COLS + CAT_COLS].isnull().sum(axis=1)
    return df

_demo = add_missing_indicators(train.drop(columns=["id", TARGET]))
new_cols = [c for c in _demo.columns if c.endswith("_missing") or c == "n_missing"]
print(f"added {len(new_cols)} columns: {new_cols}")
_demo[new_cols].describe().T[["mean", "min", "max"]]

added 13 columns: ['age_missing', 'daily_screen_time_hours_missing', 'social_media_hours_missing', 'gaming_hours_missing', 'work_study_hours_missing', 'sleep_hours_missing', 'notifications_per_day_missing', 'app_opens_per_day_missing', 'weekend_screen_time_missing', 'gender_missing', 'stress_level_missing', 'academic_work_impact_missing', 'n_missing']


,mean,min,max
age_missing,0.041843,0.0,1.0
daily_screen_time_hours_missing,0.138644,0.0,1.0
social_media_hours_missing,0.193811,0.0,1.0
gaming_hours_missing,0.183435,0.0,1.0
work_study_hours_missing,0.074516,0.0,1.0
sleep_hours_missing,0.064336,0.0,1.0
notifications_per_day_missing,0.097754,0.0,1.0
app_opens_per_day_missing,0.116739,0.0,1.0
weekend_screen_time_missing,0.162089,0.0,1.0
gender_missing,0.041995,0.0,1.0


In [7]:
def add_ratio_features(df):
    df = df.copy()
    eps = 1e-3
    df["social_to_screen_ratio"] = df["social_media_hours"] / (df["daily_screen_time_hours"] + eps)
    df["gaming_to_screen_ratio"] = df["gaming_hours"] / (df["daily_screen_time_hours"] + eps)
    df["work_to_screen_ratio"] = df["work_study_hours"] / (df["daily_screen_time_hours"] + eps)
    df["screen_to_sleep_ratio"] = df["daily_screen_time_hours"] / (df["sleep_hours"] + eps)
    df["weekend_vs_weekday_screen"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    df["weekend_screen_ratio"] = df["weekend_screen_time"] / (df["daily_screen_time_hours"] + eps)
    df["opens_per_notification"] = df["app_opens_per_day"] / (df["notifications_per_day"] + eps)
    df["notifications_per_hour"] = df["notifications_per_day"] / (df["daily_screen_time_hours"] + eps)
    df["opens_per_hour"] = df["app_opens_per_day"] / (df["daily_screen_time_hours"] + eps)
    return df

_demo = add_ratio_features(_demo)
ratio_cols = ["social_to_screen_ratio", "gaming_to_screen_ratio", "work_to_screen_ratio",
              "screen_to_sleep_ratio", "weekend_vs_weekday_screen", "weekend_screen_ratio",
              "opens_per_notification", "notifications_per_hour", "opens_per_hour"]
_demo[ratio_cols].describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
social_to_screen_ratio,0.326801,0.134389,0.000000,0.885186
gaming_to_screen_ratio,0.197084,0.111091,0.000000,0.799316
work_to_screen_ratio,0.318492,0.137186,0.000000,0.927988
screen_to_sleep_ratio,1.159260,0.468851,0.055673,3.288752
weekend_vs_weekday_screen,1.837633,1.766462,-7.910000,11.490000
weekend_screen_ratio,1.330089,0.497962,0.085413,20.399202
opens_per_notification,1.012312,1.058622,0.060000,8.999550
notifications_per_hour,22.988328,18.694627,1.374476,499.001996
opens_per_hour,15.989907,12.829827,1.092822,359.281437


In [8]:
def add_time_budget_features(df):
    df = df.copy()
    eps = 1e-3
    df["leisure_hours"] = df["social_media_hours"].fillna(0) + df["gaming_hours"].fillna(0)
    df["leisure_to_work_ratio"] = df["leisure_hours"] / (df["work_study_hours"] + eps)
    df["total_accounted_hours"] = (
        df["social_media_hours"].fillna(0)
        + df["gaming_hours"].fillna(0)
        + df["work_study_hours"].fillna(0)
        + df["sleep_hours"].fillna(0)
    )
    df["free_hours_24"] = 24 - df["total_accounted_hours"]
    df["sleep_deficit"] = 8 - df["sleep_hours"]
    return df

_demo = add_time_budget_features(_demo)
_demo[["leisure_hours", "leisure_to_work_ratio", "total_accounted_hours",
       "free_hours_24", "sleep_deficit"]].describe().T[["mean", "min", "max"]]

,mean,min,max
leisure_hours,3.183709,0.0,11.290000
leisure_to_work_ratio,1.886198,0.0,20.909091
total_accounted_hours,11.740872,0.0,20.000000
free_hours_24,12.259128,4.0,24.000000
sleep_deficit,1.195666,-1.0,3.500000


In [9]:
def add_interaction_features(df):
    df = df.copy()
    eps = 1e-3
    df["screen_per_age"] = df["daily_screen_time_hours"] / (df["age"] + eps)
    df["notif_x_opens"] = df["notifications_per_day"] * df["app_opens_per_day"]
    df["screen_x_social"] = df["daily_screen_time_hours"] * df["social_media_hours"]
    df["screen_sq"] = df["daily_screen_time_hours"] ** 2
    df["social_sq"] = df["social_media_hours"] ** 2
    df["stress_academic_combo"] = (
        df["stress_level"].astype(str) + "_" + df["academic_work_impact"].astype(str)
    )
    return df

_demo = add_interaction_features(_demo)
print("example combo values:", _demo["stress_academic_combo"].unique()[:5])
_demo[["screen_per_age", "notif_x_opens", "screen_x_social", "screen_sq", "social_sq"]].describe().T[["mean", "min", "max"]]

example combo values: ['Medium_No' 'Low_Yes' 'Low_nan' 'High_No' 'Medium_Yes']


,mean,min,max
screen_per_age,0.297566,0.014285,0.833287
notif_x_opens,15015.364981,300.000000,45000.000000
screen_x_social,21.003304,0.000000,103.600000
screen_sq,65.789083,0.250000,225.000000
social_sq,7.838244,0.000000,64.000000


In [10]:
def engineer_features(df):
    """Full pipeline: missingness indicators -> ratio features -> time-budget features ->
    interaction/polynomial features. Applied identically to train and test."""
    df = add_missing_indicators(df)
    df = add_ratio_features(df)
    df = add_time_budget_features(df)
    df = add_interaction_features(df)
    return df

del _demo  # only needed for the step-by-step preview above

train_feat = engineer_features(train.drop(columns=["id", TARGET]))
test_feat = engineer_features(test.drop(columns=["id"]))

print(f"features before engineering: {train.shape[1] - 2}")
print(f"features after engineering:  {train_feat.shape[1]}")
print(f"train_feat shape: {train_feat.shape}, test_feat shape: {test_feat.shape}")

features before engineering: 12
features after engineering:  45
train_feat shape: (691369, 45), test_feat shape: (296302, 45)


## 5. Categorical encoding

LightGBM and XGBoost (with `enable_categorical=True`) both consume pandas `category` dtype
natively — no one-hot encoding needed. Train and test categories are unioned first so a
category seen only in test doesn't come through as an unknown/NaN level.

In [11]:
def prep_categorical(train_df, test_df, cat_cols):
    tr, te = train_df.copy(), test_df.copy()
    for c in cat_cols:
        tr[c] = tr[c].astype("category")
        te[c] = te[c].astype("category")
        cats = pd.api.types.union_categoricals([tr[c], te[c]]).categories
        tr[c] = tr[c].cat.set_categories(cats)
        te[c] = te[c].cat.set_categories(cats)
    return tr, te

all_cat_cols = CAT_COLS + ["stress_academic_combo"]
prep_tr, prep_te = prep_categorical(train_feat, test_feat, all_cat_cols)

for c in all_cat_cols:
    print(f"{c}: {len(prep_tr[c].cat.categories)} categories -> {list(prep_tr[c].cat.categories)[:6]}"
          + (" ..." if len(prep_tr[c].cat.categories) > 6 else ""))

gender: 3 categories -> ['Female', 'Male', 'Other']
stress_level: 3 categories -> ['High', 'Low', 'Medium']
academic_work_impact: 2 categories -> ['No', 'Yes']
stress_academic_combo: 12 categories -> ['High_No', 'High_Yes', 'High_nan', 'Low_No', 'Low_Yes', 'Low_nan'] ...


## 6. Train LightGBM + XGBoost with 5-fold CV

For each of the 5 folds: fit LightGBM and XGBoost on the training rows, early-stop on the
held-out fold's AUC, store out-of-fold (OOF) predictions for the validation rows, and
accumulate test-set predictions (averaged across folds). This is the slow part of the
notebook — expect roughly 30 minutes total for all 10 fits (2 models × 5 folds) on ~690K
rows.

In [12]:
lgbm_params = dict(
    objective="binary", metric="auc", boosting_type="gbdt",
    n_estimators=2500, learning_rate=0.035, num_leaves=63,
    min_child_samples=50, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)

xgb_params = dict(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=2500, learning_rate=0.035, max_depth=7,
    min_child_weight=10, subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.5, reg_lambda=1.0, tree_method="hist",
    enable_categorical=True, random_state=RANDOM_STATE, n_jobs=-1,
)

n_train, n_test = len(prep_tr), len(prep_te)
oof = {m: np.zeros(n_train) for m in ["lgbm", "xgb"]}
test_pred = {m: np.zeros(n_test) for m in ["lgbm", "xgb"]}
fold_aucs = {m: [] for m in ["lgbm", "xgb"]}
lgbm_models, xgb_models = [], []

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(folds):
    print(f"=== Fold {fold+1}/{N_SPLITS} ===", flush=True)
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = lgb.LGBMClassifier(**lgbm_params)
    model.fit(
        prep_tr.iloc[tr_idx], y_tr,
        eval_set=[(prep_tr.iloc[va_idx], y_va)],
        eval_metric="auc", categorical_feature=all_cat_cols,
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    oof["lgbm"][va_idx] = model.predict_proba(prep_tr.iloc[va_idx], num_iteration=model.best_iteration_)[:, 1]
    test_pred["lgbm"] += model.predict_proba(prep_te, num_iteration=model.best_iteration_)[:, 1] / N_SPLITS
    auc_l = roc_auc_score(y_va, oof["lgbm"][va_idx])
    fold_aucs["lgbm"].append(auc_l)
    lgbm_models.append(model)
    print(f"  LightGBM AUC: {auc_l:.5f} (best_iter={model.best_iteration_})", flush=True)

    xmodel = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=100)
    xmodel.fit(
        prep_tr.iloc[tr_idx], y_tr,
        eval_set=[(prep_tr.iloc[va_idx], y_va)],
        verbose=False,
    )
    oof["xgb"][va_idx] = xmodel.predict_proba(prep_tr.iloc[va_idx])[:, 1]
    test_pred["xgb"] += xmodel.predict_proba(prep_te)[:, 1] / N_SPLITS
    auc_x = roc_auc_score(y_va, oof["xgb"][va_idx])
    fold_aucs["xgb"].append(auc_x)
    xgb_models.append(xmodel)
    print(f"  XGBoost  AUC: {auc_x:.5f} (best_iter={xmodel.best_iteration})", flush=True)

print(f"\nTraining time: {time.time() - t0:.1f}s")

=== Fold 1/5 ===


  LightGBM AUC: 0.96292 (best_iter=2159)


  XGBoost  AUC: 0.96349 (best_iter=2499)


=== Fold 2/5 ===


  LightGBM AUC: 0.96344 (best_iter=1858)


...

  XGBoost  AUC: 0.96499 (best_iter=2480)


=== Fold 5/5 ===


  LightGBM AUC: 0.96350 (best_iter=1916)


  XGBoost  AUC: 0.96387 (best_iter=2460)



Training time: 1675.2s


## 7. Per-model out-of-fold evaluation

In [13]:
fold_summary = pd.DataFrame(fold_aucs, index=[f"fold {i+1}" for i in range(N_SPLITS)])
fold_summary.loc["mean"] = fold_summary.mean()
fold_summary.loc["std"] = fold_summary.loc[[f"fold {i+1}" for i in range(N_SPLITS)]].std()
fold_summary.round(5)

,lgbm,xgb
fold 1,0.96292,0.96349
fold 2,0.96344,0.96404
fold 3,0.96397,0.96420
fold 4,0.96454,0.96499
fold 5,0.96350,0.96387
mean,0.96367,0.96412
std,0.00061,0.00056


In [14]:
print("=== Individual model OOF AUCs (full training set) ===")
for m in ["lgbm", "xgb"]:
    print(f"  {m}: {roc_auc_score(y, oof[m]):.5f}")

=== Individual model OOF AUCs (full training set) ===


  lgbm: 0.96367


  xgb: 0.96412


## 8. Stack via logistic regression (in logit space)

The meta-model learns how to weight LightGBM's and XGBoost's OOF predictions, working in
logit (log-odds) space so it's fitting a linear combination of log-odds rather than raw
probabilities — a better-behaved space for a linear model, and standard practice for
probability stacking. Fitting the meta-model on OOF predictions (not in-sample predictions)
avoids leakage: each model's OOF prediction for a row was made by a model that never saw
that row during training.

In [15]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

stack_oof = np.column_stack([logit(oof["lgbm"]), logit(oof["xgb"])])
stack_test = np.column_stack([logit(test_pred["lgbm"]), logit(test_pred["xgb"])])

meta = LogisticRegression(C=1.0, max_iter=2000)
meta.fit(stack_oof, y)
oof_blend = meta.predict_proba(stack_oof)[:, 1]
blend_auc = roc_auc_score(y, oof_blend)

print(f"Stacked blend OOF AUC: {blend_auc:.5f}")
print(f"Meta weights (lgbm, xgb): {meta.coef_[0]}, intercept: {meta.intercept_[0]:.5f}")

simple_avg = (oof["lgbm"] + oof["xgb"]) / 2
print(f"Simple average OOF AUC: {roc_auc_score(y, simple_avg):.5f}  (sanity check vs. the learned stack)")

Stacked blend OOF AUC: 0.96427
Meta weights (lgbm, xgb): [0.36654525 0.68965982], intercept: -0.01966
Simple average OOF AUC: 0.96423  (sanity check vs. the learned stack)


In [16]:
results = pd.DataFrame({
    "model": ["LightGBM", "XGBoost", "Stacked ensemble"],
    "oof_auc": [roc_auc_score(y, oof["lgbm"]), roc_auc_score(y, oof["xgb"]), blend_auc],
}).set_index("model").round(4)
results

,oof_auc
model,
LightGBM,0.9637
XGBoost,0.9641
Stacked ensemble,0.9643


## 9. Feature importance

Gain-based importance, averaged across the 5 LightGBM folds — a quick sanity check that
the top features line up with what the EDA notebooks flagged (`notifications_per_day`,
`app_opens_per_day`, `weekend_screen_time` and their engineered ratios).

In [17]:
importances = pd.DataFrame(
    {f"fold_{i+1}": m.feature_importances_ for i, m in enumerate(lgbm_models)},
    index=prep_tr.columns,
)
importances["mean"] = importances.mean(axis=1)
importances.sort_values("mean", ascending=False).head(15)[["mean"]]

,mean
notifications_per_day,10907.4
app_opens_per_day,9799.4
weekend_screen_time,7570.0
work_study_hours,5019.2
gaming_hours,4936.2
work_to_screen_ratio,4882.0
daily_screen_time_hours,4795.0
gaming_to_screen_ratio,4729.0
social_to_screen_ratio,4644.8
social_media_hours,4551.0


## 10. Predict on the test set and write the submission

Test-set predictions were already accumulated fold-by-fold in step 6 (`test_pred`), so this
just re-derives them from the same stacking meta-model and writes the submission file. This
is written to a notebook-specific filename so it doesn't overwrite
`submissions/submission_ensemble.csv` from the original script run.

In [18]:
test_blend = meta.predict_proba(stack_test)[:, 1]

submission = pd.DataFrame({"id": test_ids, "addicted_label": test_blend})
out_path = f"{OUT_DIR}/submission_ensemble_notebook.csv"
submission.to_csv(out_path, index=False)

print(f"Saved submission to {out_path}")
submission.head()

Saved submission to ../submissions/submission_ensemble_notebook.csv


,id,addicted_label
0,691369,0.999768
1,691370,0.943497
2,691371,0.969234
3,691372,0.993914
4,691373,0.998034


## Summary

- **Splitting:** a single stratified hold-out split illustrates the basic mechanics, but
  the model itself is trained with 5-fold stratified CV, which gives a stable OOF AUC
  estimate and the OOF predictions needed for stacking.
- **Feature engineering:** missingness indicators, usage ratios, time-budget features, and
  a handful of interaction/polynomial terms — all built directly from the EDA findings.
- **Models:** LightGBM and XGBoost, each 5-fold CV with early stopping and native
  categorical support, no one-hot encoding needed.
- **Stacking:** a logistic regression meta-model combines both models' OOF predictions in
  logit space, giving a small but consistent lift over either model alone or a simple
  average.

Final OOF AUCs from this run are printed in the results table in step 8 — compare against
the README's 0.9637 / 0.9641 / 0.9642 (same pipeline, same seed; expect them to match
closely, with any tiny differences coming from multi-threaded floating-point
non-determinism in LightGBM/XGBoost).